# UAE Mobile Intelligence - First UAE Map (simple version)

The simplest possible version of the Phase 1 map: hexagons tile the whole UAE (there is no
background map of the world -- the hexagons themselves form the country's shape), colored by
download speed, with upload speed / latency / test count shown as plain text when you hover a
hexagon. Only download speed is colored for now -- upload and latency are along for the ride as
extra numbers, not extra colors, to keep this first version easy to follow.

Uses **H3 resolution 7** -- the same resolution chosen and justified for scoring in
[`04_h3_resolution_choice.ipynb`](04_h3_resolution_choice.ipynb). An earlier version of this
notebook tried resolution 8 (smaller hexagons) for a crisper picture, but resolution 8 covering
the *entire* UAE is ~109,900 individual hexagons -- that's 109,900 interactive shapes in one map,
which produces a 50 MB HTML file that won't render in a browser. Resolution 7 cuts that to
~15,700 hexagons, a size a browser can actually handle, and also means this map now uses the
exact same zones as every other notebook in the pipeline.

Steps, in order:
1. Build a (bigger) hexagon for every part of the UAE.
2. Aggregate the raw Ookla tiles into those hexagons: download, upload, latency, test count.
3. Pick a color for each hexagon, based on download speed only.
4. Draw the map and save it.

In [1]:
import json
from pathlib import Path

import folium
import geopandas as gpd
import h3
import matplotlib
import matplotlib.colors as mcolors
import pandas as pd

In [2]:
# Settings -- change these to explore different quarters, a bigger/smaller hexagon, or a
# stricter/looser evidence bar

QUARTER_TO_SHOW = "2026Q2"
MAP_H3_RESOLUTION = 7  # matches every other notebook -- resolution 8 for the whole UAE is ~109,900 hexagons, too many to render
MIN_TESTS = 30  

# print("Map hexagon resolution:", MAP_H3_RESOLUTION)

## Step 1 -- build a hexagon for every part of the UAE

`h3.polygon_to_cells` fills a polygon with H3 hexagons. It needs `(lat, lon)` points (the UAE
boundary file stores `(lon, lat)`, so we swap them), and the UAE boundary is a *MultiPolygon* (the
mainland plus a number of small islands), so we build one `h3.LatLngPoly` per part first.

In [3]:
uae_boundary = gpd.read_file("../data/raw/boundary/uae_boundary.geojson").to_crs("EPSG:4326")
uae_shape = uae_boundary.geometry.iloc[0]  # one MultiPolygon covering the whole country

h3_polygons = []
for part in uae_shape.geoms:
    outer_ring = [(lat, lon) for lon, lat in part.exterior.coords]
    h3_polygons.append(h3.LatLngPoly(outer_ring))

uae_h3_shape = h3.LatLngMultiPoly(*h3_polygons)
all_hexagons = h3.polygon_to_cells(uae_h3_shape, MAP_H3_RESOLUTION)

print("Hexagons covering the UAE:", len(all_hexagons))

Hexagons covering the UAE: 15695


## Step 2 -- aggregate the raw Ookla tiles into these hexagons

`05_zone_aggregation.ipynb` already built this table at resolution 7. This map uses the same H3 resolution 7, but for one quarter only, so we repeat the same
simple aggregation here rather than filtering `zone_quarter_table.parquet` down to one quarter: assign every tile to a hexagon, then test-weight the average within each hexagon (a
tile with 50 tests should count more than a tile with 1).

In [4]:
tiles = gpd.read_parquet("../data/processed/ookla_tiles_uae.parquet")
tiles = pd.DataFrame(tiles.drop(columns=["geometry", "tile_geometry"]))
tiles = tiles[tiles["quarter"] == QUARTER_TO_SHOW]

# h3 wants (lat, lon) = (tile_y, tile_x) -- the same swap gotcha as everywhere else in this dataset
tiles["h3_cell"] = [
    h3.latlng_to_cell(lat, lon, MAP_H3_RESOLUTION)
    for lat, lon in zip(tiles["tile_y"], tiles["tile_x"])
]

tiles["w_download"] = tiles["avg_d_kbps"] * tiles["tests"]
tiles["w_upload"] = tiles["avg_u_kbps"] * tiles["tests"]
tiles["w_latency"] = tiles["avg_lat_down_ms"] * tiles["tests"]

zones = tiles.groupby("h3_cell").agg(
    tests=("tests", "sum"),
    w_download=("w_download", "sum"),
    w_upload=("w_upload", "sum"),
    w_latency=("w_latency", "sum"),
).reset_index()

zones["download_mbps"] = zones["w_download"] / zones["tests"] / 1000
zones["upload_mbps"] = zones["w_upload"] / zones["tests"] / 1000
zones["latency_ms"] = zones["w_latency"] / zones["tests"]

print("Hexagons with at least one measurement:", len(zones))
zones[["h3_cell", "tests", "download_mbps", "upload_mbps", "latency_ms"]].head()

Hexagons with at least one measurement: 1815


,h3_cell,tests,download_mbps,upload_mbps,latency_ms
0,874384508ffffff,1,258.200000,10.086000,433.0
1,874384566ffffff,3,259.381667,16.407667,505.666667
2,874384930ffffff,1,38.532000,11.020000,323.0
3,874384ca6ffffff,1,613.907000,63.610000,604.0
4,874384d86ffffff,1,9.415000,2.891000,1503.0


## Step 3 -- turn that table into simple lookups, and pick a color

Plain Python dictionaries, one per value, so any hexagon's numbers can be looked up by its ID.
Red = slow, green = fast, using download speed between 50 and 600 Mbps as the color range -- but
only download gets a color. Upload and latency are just numbers we show in the tooltip.

In [5]:
download_by_hexagon = dict(zip(zones["h3_cell"], zones["download_mbps"]))
upload_by_hexagon = dict(zip(zones["h3_cell"], zones["upload_mbps"]))
latency_by_hexagon = dict(zip(zones["h3_cell"], zones["latency_ms"]))
tests_by_hexagon = dict(zip(zones["h3_cell"], zones["tests"]))

color_scale = matplotlib.colormaps["RdYlGn"]
speed_range = mcolors.Normalize(vmin=50, vmax=600)
GREY_NOT_ENOUGH_DATA = "#e0e0e0"


def color_for_hexagon(hexagon_id):
    tests = tests_by_hexagon.get(hexagon_id, 0)
    if tests < MIN_TESTS:
        return GREY_NOT_ENOUGH_DATA
    download_speed = download_by_hexagon[hexagon_id]
    return mcolors.rgb2hex(color_scale(speed_range(download_speed)))


def label_for_hexagon(hexagon_id):
    tests = tests_by_hexagon.get(hexagon_id, 0)
    if tests < MIN_TESTS:
        return f"Not enough measurements ({tests} tests)"
    return (
        f"Download: {download_by_hexagon[hexagon_id]:.0f} Mbps | "
        f"Upload: {upload_by_hexagon[hexagon_id]:.0f} Mbps | "
        f"Latency: {latency_by_hexagon[hexagon_id]:.0f} ms | "
        f"Tests: {tests}"
    )


# quick check on one hexagon
sample_hexagon = next(iter(download_by_hexagon))
print(sample_hexagon, "->", color_for_hexagon(sample_hexagon))
print(label_for_hexagon(sample_hexagon))

874384508ffffff -> #e0e0e0
Not enough measurements (1 tests)


## Step 4 -- draw the map

One shape (a GeoJSON "feature") per hexagon, each carrying its own color and label. `tiles=None`
means there's no background map of the world -- the hexagons are the only thing drawn, so their
combined outline is what shows the UAE.

In [6]:
hexagon_shapes = []

for hexagon_id in all_hexagons:
    corners = h3.cell_to_boundary(hexagon_id)  # list of (lat, lon) points
    ring = [[lon, lat] for lat, lon in corners]  # GeoJSON wants (lon, lat)
    ring.append(ring[0])  # close the ring (last point = first point)

    hexagon_shapes.append({
        "type": "Feature",
        "geometry": {"type": "Polygon", "coordinates": [ring]},
        "properties": {
            "color": color_for_hexagon(hexagon_id),
            "label": label_for_hexagon(hexagon_id),
        },
    })

hexagon_grid = {"type": "FeatureCollection", "features": hexagon_shapes}
print("Shapes to draw:", len(hexagon_shapes))

Shapes to draw:

 15695


In [7]:
m = folium.Map(tiles=None, zoom_control=True, prefer_canvas=True)

folium.GeoJson(
    hexagon_grid,
    style_function=lambda feature: {
        "fillColor": feature["properties"]["color"],
        "color": "#999999",  # thin grey outline between hexagons
        "weight": 0.5,
        "fillOpacity": 0.85,
    },
    tooltip=folium.GeoJsonTooltip(fields=["label"], aliases=["Measurements"]),
).add_to(m)

# zoom to fit the UAE exactly, instead of showing the whole world
min_lon, min_lat, max_lon, max_lat = uae_shape.bounds
m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])

m

## Save as a standalone HTML

So the map can be opened directly in a browser without re-running the notebook.

In [8]:
out_path = Path("../data/processed/uae_map_" + QUARTER_TO_SHOW + ".html")
m.save(str(out_path))
print(f"Saved: {out_path} ({out_path.stat().st_size / 1024:.1f} KB)")

Saved: ..\data\processed\uae_map_2026Q2.html (7214.1 KB)


## What this map does and doesn't show

- **Shows:** download speed (colored), plus upload speed, latency and test count (as text in the
  hover tooltip) for every hexagon in the UAE, for one quarter (2026Q2). Grey hexagons have no
  measurements at all this quarter -- there's nothing to color. Every hexagon with at least one
  test is colored, however few tests it has -- test count isn't used for confidence yet, that
  comes later as its own dedicated step (the brief's Confidence Score).
- **Doesn't show yet:** its own colors for upload/latency, population, or the
  Experience/Confidence/Priority scores -- kept out on purpose to keep this version simple. Once
  this is comfortable, the same pattern (a dictionary of hexagon -> value, a color function, one
  GeoJson layer) can be repeated to color by a different metric, or to add a layer switcher.

This clears the Phase 1 gate: *"You can correctly display real public UAE mobile measurements on a
map."* Next: OSM UAE feature extraction, the data dictionary, and the T0 coverage audit -- see the
repo README for the updated next-steps list.